# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAlmasri7/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

**Rule:** I will rank pages that have enough Google Search Console impressions to provide meaningful evidence and whose CTR is lower than the typical CTR observed for pages in a similar average-position range. A larger CTR gap combined with stronger search visibility will result in a higher priority score.

**Reason code:** `CTR_BELOW_POSITION_PEERS` — the page's observed CTR is below the typical CTR for pages in a similar position range.

**Action:** `REVIEW_SNIPPET` — review the page's search snippet and intent alignment.

This rule is intended for **decision support**, not causal inference. A low CTR does not prove that the title, meta description, or content is the cause. Instead, it identifies pages that are worth reviewing based on the observed search signals.

The rule will use only signals available at the decision moment and will not use future-window metrics, label-derived fields, or existing product flags.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# ------------------------------------------------------------
# 1. Load HF_TOKEN from .env
# ------------------------------------------------------------

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Make sure you have a .env file "
        "containing: HF_TOKEN=your_token"
    )

print("Hugging Face token loaded successfully.")


# ------------------------------------------------------------
# 2. Connect to DuckDB
# ------------------------------------------------------------

con = duckdb.connect()

# Create the Hugging Face secret using the token
con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection established.")


# ------------------------------------------------------------
# 3. Warehouse location
# ------------------------------------------------------------

rel = "hf://datasets/FlyRank/internship-warehouse"

# We will use the daily performance fact table.
table_path = (
    f"{rel}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print("Using:", table_path)

Hugging Face token loaded successfully.
DuckDB connection established.
Using: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [2]:
# ------------------------------------------------------------
# 4. Check the March 2026 slice
# ------------------------------------------------------------

march_summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{table_path}')
""").df()

march_summary

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [3]:
# ------------------------------------------------------------
# 5. Inspect the available columns
# ------------------------------------------------------------

columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{table_path}')
""").df()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [4]:
# ------------------------------------------------------------
# 6. Create the analysis dataset
# ------------------------------------------------------------

march_df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_avg_position
FROM read_parquet('{table_path}')
WHERE gsc_impressions > 0
  AND gsc_avg_position > 0
""").df()

print("Eligible rows:", len(march_df))

march_df.head()

Eligible rows: 3447872


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,4.928000
2,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,4.000000
3,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,2.272727
4,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,1756,7.347280


In [6]:
position_check = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position > 0 AND gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20 THEN '11-20'
        WHEN gsc_avg_position > 20 AND gsc_avg_position <= 50 THEN '21-50'
        WHEN gsc_avg_position > 50 THEN '50+'
    END AS position_group,

    COUNT(*) AS n,

    SUM(gsc_impressions) AS impressions,

    SUM(gsc_clicks) AS clicks,

    ROUND(
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0),
        4
    ) AS aggregated_ctr,

    ROUND(
        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_clicks * 1.0 / gsc_impressions
            END
        ),
        4
    ) AS mean_page_ctr,

    ROUND(
        AVG(
            CASE
                WHEN gsc_clicks > 0 THEN 1.0
                ELSE 0.0
            END
        ),
        4
    ) AS pct_pages_with_click

FROM read_parquet('{table_path}')

WHERE gsc_impressions > 0
  AND gsc_avg_position > 0

GROUP BY position_group

ORDER BY
    CASE position_group
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21-50' THEN 4
        WHEN '50+' THEN 5
    END
""").df()

position_check

,position_group,n,impressions,clicks,aggregated_ctr,mean_page_ctr,pct_pages_with_click
0,1-3,564173,53559848.0,204283.0,0.0038,0.0049,0.1622
1,4-10,1456122,137830113.0,445828.0,0.0032,0.0035,0.1522
2,11-20,519223,29386006.0,92449.0,0.0031,0.0028,0.1031
3,21-50,631491,55944412.0,76589.0,0.0014,0.0016,0.0773
4,50+,276863,3468464.0,1509.0,0.0004,0.0005,0.0048


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.